# Rathipriya et al. (2023) — Daily PharmaSales Baseline Comparison

This notebook compares two preprocessing approaches for the Rathipriya et al. (2023) pharma daily sales baselines:

**Reference-Based Exploration** — follows the paper's methodology: chronological 70/15/15 split (train/validation/test), single `lag_1` feature, hyperparameter tuning on validation set, final evaluation on test set.

**Our Preprocessing** — uses our own pipeline: chronological 80/20 split (train/test), ACF-based lag selection (`max_lag=26`) + rolling mean features, grid search with evaluation on test set.

**Models:** GRNN, P_NN, RBFNN, and ARIMA evaluated in both sections.
**Metrics:** MSE, RMSE, and normalized RMSE.


In [118]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.cluster import KMeans
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_squared_error, root_mean_squared_error
from statsmodels.tsa.stattools import acf

try:
    from statsmodels.tsa.arima.model import ARIMA
    STATSMODELS_AVAILABLE = True
except Exception:
    ARIMA = None
    STATSMODELS_AVAILABLE = False

CATEGORIES = ["M01AB", "M01AE", "N02BA", "N02BE", "N05B", "N05C", "R03", "R06"]
MAX_LAG = 26
RANDOM_STATE = 42

BANDWIDTHS = [0.05, 0.1, 0.2, 0.5, 1.0, 2.0, 5.0]
RBF_CENTERS = [5, 10, 20, 40]
RBF_GAMMAS = [0.01, 0.05, 0.1, 0.5, 1.0]
RBF_ALPHAS = [0.01, 0.1, 1.0]

# Model Architectures

In [119]:
def normalized_rmse(y_true, rmse):
    mean_y = float(np.mean(y_true))
    return rmse / mean_y if mean_y else np.nan


def gaussian_kernel_predict(X_train, y_train, X_pred, sigma, batch_size=256):
    predictions = []
    sigma = max(float(sigma), 1e-12)

    for start in range(0, len(X_pred), batch_size):
        batch = X_pred[start:start + batch_size]
        distances_sq = ((batch[:, None, :] - X_train[None, :, :]) ** 2).sum(axis=2)
        weights = np.exp(-distances_sq / (2 * sigma ** 2))
        denom = weights.sum(axis=1)
        fallback = np.full(len(batch), np.mean(y_train), dtype=float)
        predictions.append(np.divide(weights @ y_train, denom, out=fallback, where=denom > 1e-12))

    return np.concatenate(predictions)


def pnn_predict(X_train, y_train, X_pred, sigma, n_bins=20, batch_size=256):
    y_min, y_max = y_train.min(), y_train.max()
    margin = (y_max - y_min) * 0.01 if y_max > y_min else 1.0
    y_min -= margin
    y_max += margin
    bins = np.linspace(y_min, y_max, n_bins + 1)
    bin_centers = (bins[:-1] + bins[1:]) / 2
    bin_idx = np.digitize(y_train, bins) - 1
    bin_idx = np.clip(bin_idx, 0, n_bins - 1)

    predictions = []
    sigma = max(float(sigma), 1e-12)

    for start in range(0, len(X_pred), batch_size):
        batch = X_pred[start:start + batch_size]
        distances_sq = ((batch[:, None, :] - X_train[None, :, :]) ** 2).sum(axis=2)
        weights = np.exp(-distances_sq / (2 * sigma ** 2))
        batch_preds = np.zeros(len(batch))
        for j in range(len(batch)):
            bin_sums = np.bincount(bin_idx, weights=weights[j], minlength=n_bins)
            batch_preds[j] = bin_centers[np.argmax(bin_sums)]
        predictions.append(batch_preds)

    return np.concatenate(predictions)


def rbf_features(X, centers, gamma):
    distances_sq = ((X[:, None, :] - centers[None, :, :]) ** 2).sum(axis=2)
    return np.exp(-float(gamma) * distances_sq)


def fit_predict_rbfnn(X_train, y_train, X_pred, n_centers, gamma, alpha):
    n_centers = min(int(n_centers), len(X_train))
    kmeans = KMeans(n_clusters=n_centers, random_state=RANDOM_STATE, n_init=10)
    centers = kmeans.fit(X_train).cluster_centers_
    model = Ridge(alpha=alpha)
    model.fit(rbf_features(X_train, centers, gamma), y_train)
    return model.predict(rbf_features(X_pred, centers, gamma))

# Reference-Based Exploration

Exploration follows the methodology from Rathipriya et al. (2023). Preprocessing: chronological 70/15/15 split (train/validation/test), basic `lag_1` feature (without ACF selection and rolling mean). Hyperparameter tuning is performed on the validation set, final evaluation on the test set.


## Preprocessing

Preprocessing follows the paper: data is loaded, then split chronologically 70% train / 15% validation / 15% test. Only `lag_1` (y(t-1)) is used as feature per category — without ACF-based lag selection and rolling mean, unlike our preprocessing in the next section.


In [120]:
ref_categories = []

for category in CATEGORIES:
    dfg = df[["datum", category]].rename(columns={"datum": "ds", category: "y"}).copy()
    dfg["lag_1"] = dfg["y"].shift(1)
    dfg = dfg.dropna().reset_index(drop=True)

    train_size = int(len(dfg) * 0.70)
    val_size = int(len(dfg) * 0.15)

    train_df = dfg.iloc[:train_size]
    val_df = dfg.iloc[train_size:train_size + val_size]
    test_df = dfg.iloc[train_size + val_size:]

    feature_columns = ["lag_1"]

    ref_categories.append({
        "Category": category,
        "feature_columns": feature_columns,
        "X_train": train_df[feature_columns].to_numpy(dtype=float),
        "y_train": train_df["y"].to_numpy(dtype=float),
        "X_val": val_df[feature_columns].to_numpy(dtype=float),
        "y_val": val_df["y"].to_numpy(dtype=float),
        "X_test": test_df[feature_columns].to_numpy(dtype=float),
        "y_test": test_df["y"].to_numpy(dtype=float),
    })

pd.DataFrame([
    {
        "Category": item["Category"],
        "train_size": len(item["X_train"]),
        "val_size": len(item["X_val"]),
        "test_size": len(item["X_test"]),
    }
    for item in ref_categories
])


,Category,train_size,val_size,test_size
0,M01AB,1473,315,317
1,M01AE,1473,315,317
2,N02BA,1473,315,317
3,N02BE,1473,315,317
4,N05B,1473,315,317
5,N05C,1473,315,317
6,R03,1473,315,317
7,R06,1473,315,317


## Modeling

SNN models (GRNN, P_NN, RBFNN) and ARIMA are run with reference preprocessing (70/15/15 split, `lag_1` feature). Hyperparameter tuning is performed on the validation set, final evaluation on the test set.


### GRNN

General Regression Neural Network — kernel regression with Gaussian kernel. Hyperparameter tuning (`sigma`) is performed on the validation set. The best model is evaluated on the test set.


In [121]:
ref_grnn_records = []

for item in ref_categories:
    category = item["Category"]
    X_train = item["X_train"]
    y_train = item["y_train"]
    X_val = item["X_val"]
    y_val = item["y_val"]
    X_test = item["X_test"]
    y_test = item["y_test"]

    print(f"Running {category}...")

    # --- Training & Validation: Grid Search ---
    best_grnn = None
    for sigma in BANDWIDTHS:
        pred = gaussian_kernel_predict(X_train, y_train, X_val, sigma)
        pred = np.maximum(pred, 0)
        mse = float(mean_squared_error(y_val, pred))
        rmse = float(root_mean_squared_error(y_val, pred))
        if best_grnn is None or rmse < best_grnn["Val RMSE"]:
            best_grnn = {"sigma": sigma, "Val MSE": mse, "Val RMSE": rmse}

    # --- Testing: Best Model on Test Set ---
    pred_test = gaussian_kernel_predict(X_train, y_train, X_test, best_grnn["sigma"])
    pred_test = np.maximum(pred_test, 0)
    test_mse = float(mean_squared_error(y_test, pred_test))
    test_rmse = float(root_mean_squared_error(y_test, pred_test))

    ref_grnn_records.append({
        "Category": category,
        "Method": "GRNN",
        "params": {"sigma": best_grnn["sigma"]},
        "Val MSE": best_grnn["Val MSE"],
        "Val RMSE": best_grnn["Val RMSE"],
        "Test MSE": test_mse,
        "Test RMSE": test_rmse,
        "normalized_rmse": normalized_rmse(y_test, test_rmse),
    })

ref_grnn_df = pd.DataFrame(ref_grnn_records)


Running M01AB...
Running M01AE...
Running N02BA...
Running N02BE...
Running N05B...
Running N05C...
Running R03...
Running R06...


### P_NN

Probabilistic Neural Network — 4-layer architecture (input/pattern/summation/output). Hyperparameter tuning (`sigma`) on the validation set.


In [122]:
ref_pnn_records = []

for item in ref_categories:
    category = item["Category"]
    X_train = item["X_train"]
    y_train = item["y_train"]
    X_val = item["X_val"]
    y_val = item["y_val"]
    X_test = item["X_test"]
    y_test = item["y_test"]

    print(f"Running {category}...")

    # --- Training & Validation: Grid Search ---
    best_pnn = None
    for sigma in BANDWIDTHS:
        pred = pnn_predict(X_train, y_train, X_val, sigma)
        pred = np.maximum(pred, 0)
        mse = float(mean_squared_error(y_val, pred))
        rmse = float(root_mean_squared_error(y_val, pred))
        if best_pnn is None or rmse < best_pnn["Val RMSE"]:
            best_pnn = {"sigma": sigma, "Val MSE": mse, "Val RMSE": rmse}

    # --- Testing: Best Model on Test Set ---
    pred_test = pnn_predict(X_train, y_train, X_test, best_pnn["sigma"])
    pred_test = np.maximum(pred_test, 0)
    test_mse = float(mean_squared_error(y_test, pred_test))
    test_rmse = float(root_mean_squared_error(y_test, pred_test))

    ref_pnn_records.append({
        "Category": category,
        "Method": "P_NN",
        "params": {"sigma": best_pnn["sigma"]},
        "Val MSE": best_pnn["Val MSE"],
        "Val RMSE": best_pnn["Val RMSE"],
        "Test MSE": test_mse,
        "Test RMSE": test_rmse,
        "normalized_rmse": normalized_rmse(y_test, test_rmse),
    })

ref_pnn_df = pd.DataFrame(ref_pnn_records)


Running M01AB...
Running M01AE...
Running N02BA...
Running N02BE...
Running N05B...
Running N05C...
Running R03...
Running R06...


### RBFNN

Radial Basis Function Neural Network — K-Means clustering + Ridge regression. Hyperparameter tuning on the validation set.


In [123]:
ref_rbfnn_records = []

for item in ref_categories:
    category = item["Category"]
    X_train = item["X_train"]
    y_train = item["y_train"]
    X_val = item["X_val"]
    y_val = item["y_val"]
    X_test = item["X_test"]
    y_test = item["y_test"]

    print(f"Running {category}...")

    # --- Validation: Grid Search (Training + Eval on Val) ---
    best_rbfnn = None
    for n_centers in [n for n in RBF_CENTERS if n <= len(X_train)]:
        for gamma in RBF_GAMMAS:
            for alpha in RBF_ALPHAS:
                # Training: fit on X_train, predict on X_val
                pred = fit_predict_rbfnn(X_train, y_train, X_val, n_centers, gamma, alpha)
                pred = np.maximum(pred, 0)
                mse = float(mean_squared_error(y_val, pred))
                rmse = float(root_mean_squared_error(y_val, pred))
                if best_rbfnn is None or rmse < best_rbfnn["Val RMSE"]:
                    best_rbfnn = {"n_centers": n_centers, "gamma": gamma, "alpha": alpha, "Val MSE": mse, "Val RMSE": rmse}

    # --- Testing: Best Model on Test Set ---
    pred_test = fit_predict_rbfnn(X_train, y_train, X_test, best_rbfnn["n_centers"], best_rbfnn["gamma"], best_rbfnn["alpha"])
    pred_test = np.maximum(pred_test, 0)
    test_mse = float(mean_squared_error(y_test, pred_test))
    test_rmse = float(root_mean_squared_error(y_test, pred_test))

    ref_rbfnn_records.append({
        "Category": category,
        "Method": "RBFNN",
        "params": {"n_centers": best_rbfnn["n_centers"], "gamma": best_rbfnn["gamma"], "alpha": best_rbfnn["alpha"]},
        "Val MSE": best_rbfnn["Val MSE"],
        "Val RMSE": best_rbfnn["Val RMSE"],
        "Test MSE": test_mse,
        "Test RMSE": test_rmse,
        "normalized_rmse": normalized_rmse(y_test, test_rmse),
    })

ref_rbfnn_df = pd.DataFrame(ref_rbfnn_records)


Running M01AB...
Running M01AE...
Running N02BA...
Running N02BE...
Running N05B...
Running N05C...


d:\Work-Env\ITEC\forecasting-medicine-public\.venv\Lib\site-packages\sklearn\base.py:1403: ConvergenceWarning: Number of distinct clusters (19) found smaller than n_clusters (20). Possibly due to duplicate points in X.
  return fit_method(estimator, *args, **kwargs)
d:\Work-Env\ITEC\forecasting-medicine-public\.venv\Lib\site-packages\sklearn\base.py:1403: ConvergenceWarning: Number of distinct clusters (19) found smaller than n_clusters (20). Possibly due to duplicate points in X.
  return fit_method(estimator, *args, **kwargs)
d:\Work-Env\ITEC\forecasting-medicine-public\.venv\Lib\site-packages\sklearn\base.py:1403: ConvergenceWarning: Number of distinct clusters (19) found smaller than n_clusters (20). Possibly due to duplicate points in X.
  return fit_method(estimator, *args, **kwargs)
d:\Work-Env\ITEC\forecasting-medicine-public\.venv\Lib\site-packages\sklearn\base.py:1403: ConvergenceWarning: Number of distinct clusters (19) found smaller than n_clusters (20). Possibly due to dup

Running R03...
Running R06...


### ARIMA

Autoregressive Integrated Moving Average — statistical time series model with order (5,1,0). ARIMA only uses the y time-series data (no feature matrix).


In [124]:
ref_arima_records = []

if STATSMODELS_AVAILABLE and ARIMA is not None:
    for item in ref_categories:
        category = item["Category"]
        y_train = item["y_train"]
        y_val = item["y_val"]
        y_test = item["y_test"]

        print(f"Running {category}...")

        # --- Training: Fit ARIMA on Train+Validation ---
        y_train_combined = np.concatenate([y_train, y_val])
        fitted = ARIMA(y_train_combined, order=(5, 1, 0)).fit()

        # --- Testing: Forecast on Test Set ---
        pred = fitted.forecast(steps=len(y_test))
        pred = np.maximum(pred, 0)
        mse = float(mean_squared_error(y_test, pred))
        rmse = float(root_mean_squared_error(y_test, pred))

        ref_arima_records.append({
            "Category": category,
            "Method": "ARIMA",
            "params": {"order": (5, 1, 0)},
            "Val MSE": np.nan,
            "Val RMSE": np.nan,
            "Test MSE": mse,
            "Test RMSE": rmse,
            "normalized_rmse": normalized_rmse(y_test, rmse) if not np.isnan(rmse) else np.nan,
        })
else:
    for item in ref_categories:
        ref_arima_records.append({
            "Category": item["Category"],
            "Method": "ARIMA",
            "params": {"order": (5, 1, 0)},
            "Val MSE": np.nan,
            "Val RMSE": np.nan,
            "Test MSE": np.nan,
            "Test RMSE": np.nan,
            "normalized_rmse": np.nan,
        })

ref_arima_df = pd.DataFrame(ref_arima_records)


Running M01AB...
Running M01AE...
Running N02BA...
Running N02BE...
Running N05B...
Running N05C...
Running R03...
Running R06...


## Reference Results

Combine all reference model results into one DataFrame for comparison with our preprocessing results.


In [125]:
ref_results = pd.concat([ref_grnn_df, ref_pnn_df, ref_rbfnn_df, ref_arima_df], ignore_index=True)
ref_results["split"] = "chronological 70/15/15; tuning on val, eval on test"
ref_results["target_scale"] = "original daily sales scale; no target scaling"
ref_results["metric_family"] = "MSE/RMSE"

ref_results.sort_values(["Category", "Test RMSE"])


,Category,Method,params,Val MSE,Val RMSE,Test MSE,Test RMSE,normalized_rmse,split,target_scale,metric_family
0,M01AB,GRNN,{'sigma': 5.0},7.257819,2.694034,8.504001,2.916162,0.548086,"chronological 70/15/15; tuning on val, eval on...",original daily sales scale; no target scaling,MSE/RMSE
16,M01AB,RBFNN,"{'n_centers': 5, 'gamma': 0.01, 'alpha': 0.01}",7.333866,2.708111,8.562604,2.926193,0.549971,"chronological 70/15/15; tuning on val, eval on...",original daily sales scale; no target scaling,MSE/RMSE
24,M01AB,ARIMA,"{'order': (5, 1, 0)}",NaN,NaN,8.959265,2.993203,0.562565,"chronological 70/15/15; tuning on val, eval on...",original daily sales scale; no target scaling,MSE/RMSE
8,M01AB,P_NN,{'sigma': 0.5},8.120519,2.849652,10.138871,3.184159,0.598455,"chronological 70/15/15; tuning on val, eval on...",original daily sales scale; no target scaling,MSE/RMSE
17,M01AE,RBFNN,"{'n_centers': 5, 'gamma': 1.0, 'alpha': 1.0}",3.999825,1.999956,5.280834,2.298006,0.598306,"chronological 70/15/15; tuning on val, eval on...",original daily sales scale; no target scaling,MSE/RMSE
1,M01AE,GRNN,{'sigma': 5.0},3.980009,1.994996,5.481097,2.341174,0.609546,"chronological 70/15/15; tuning on val, eval on...",original daily sales scale; no target scaling,MSE/RMSE
9,M01AE,P_NN,{'sigma': 2.0},4.001316,2.000329,5.554755,2.356853,0.613628,"chronological 70/15/15; tuning on val, eval on...",original daily sales scale; no target scaling,MSE/RMSE
25,M01AE,ARIMA,"{'order': (5, 1, 0)}",NaN,NaN,6.196146,2.489206,0.648087,"chronological 70/15/15; tuning on val, eval on...",original daily sales scale; no target scaling,MSE/RMSE
26,N02BA,ARIMA,"{'order': (5, 1, 0)}",NaN,NaN,4.144145,2.035717,0.664219,"chronological 70/15/15; tuning on val, eval on...",original daily sales scale; no target scaling,MSE/RMSE
10,N02BA,P_NN,{'sigma': 0.5},5.272922,2.296284,4.840193,2.200044,0.717836,"chronological 70/15/15; tuning on val, eval on...",original daily sales scale; no target scaling,MSE/RMSE


## Best Baseline Per Category (Reference)


In [126]:
ref_best = (
    ref_results
    .loc[ref_results.groupby("Category")["Test RMSE"].idxmin()]
    .reset_index(drop=True)
)
ref_best


,Category,Method,params,Val MSE,Val RMSE,Test MSE,Test RMSE,normalized_rmse,split,target_scale,metric_family
0,M01AB,GRNN,{'sigma': 5.0},7.257819,2.694034,8.504001,2.916162,0.548086,"chronological 70/15/15; tuning on val, eval on...",original daily sales scale; no target scaling,MSE/RMSE
1,M01AE,RBFNN,"{'n_centers': 5, 'gamma': 1.0, 'alpha': 1.0}",3.999825,1.999956,5.280834,2.298006,0.598306,"chronological 70/15/15; tuning on val, eval on...",original daily sales scale; no target scaling,MSE/RMSE
2,N02BA,ARIMA,"{'order': (5, 1, 0)}",NaN,NaN,4.144145,2.035717,0.664219,"chronological 70/15/15; tuning on val, eval on...",original daily sales scale; no target scaling,MSE/RMSE
3,N02BE,GRNN,{'sigma': 5.0},153.886122,12.405085,205.351803,14.330101,0.486175,"chronological 70/15/15; tuning on val, eval on...",original daily sales scale; no target scaling,MSE/RMSE
4,N05B,ARIMA,"{'order': (5, 1, 0)}",NaN,NaN,18.711654,4.325697,0.507568,"chronological 70/15/15; tuning on val, eval on...",original daily sales scale; no target scaling,MSE/RMSE
5,N05C,GRNN,{'sigma': 5.0},1.672601,1.293291,1.307072,1.143273,1.603617,"chronological 70/15/15; tuning on val, eval on...",original daily sales scale; no target scaling,MSE/RMSE
6,R03,RBFNN,"{'n_centers': 10, 'gamma': 0.5, 'alpha': 0.01}",54.746789,7.399107,70.986910,8.425373,1.145936,"chronological 70/15/15; tuning on val, eval on...",original daily sales scale; no target scaling,MSE/RMSE
7,R06,RBFNN,"{'n_centers': 5, 'gamma': 0.05, 'alpha': 0.1}",7.020337,2.649592,6.644371,2.577668,0.716038,"chronological 70/15/15; tuning on val, eval on...",original daily sales scale; no target scaling,MSE/RMSE


### RMSE (Reference)


In [127]:
# Best results per category (long format — zero NaN)
ref_table = ref_best[["Category", "Method", "Val RMSE", "Val MSE", "Test RMSE", "Test MSE"]].reset_index(drop=True)
ref_table


,Category,Method,Val RMSE,Val MSE,Test RMSE,Test MSE
0,M01AB,GRNN,2.694034,7.257819,2.916162,8.504001
1,M01AE,RBFNN,1.999956,3.999825,2.298006,5.280834
2,N02BA,ARIMA,NaN,NaN,2.035717,4.144145
3,N02BE,GRNN,12.405085,153.886122,14.330101,205.351803
4,N05B,ARIMA,NaN,NaN,4.325697,18.711654
5,N05C,GRNN,1.293291,1.672601,1.143273,1.307072
6,R03,RBFNN,7.399107,54.746789,8.425373,70.986910
7,R06,RBFNN,2.649592,7.020337,2.577668,6.644371


# Exploration Based on Our Preprocessing

Exploration using the preprocessing pipeline from our prior research.


## Preprocessing

Preprocessing pipeline includes daily data loading, ACF lag selection per category, feature engineering (lag features + rolling mean), and chronological split. Helper functions are also defined here for model evaluation.


In [128]:
DATA_PATH = Path("../data/raw/pharma-sales/salesdaily.csv")

df = pd.read_csv(DATA_PATH)
df["datum"] = pd.to_datetime(df["datum"])
df = df.sort_values("datum").reset_index(drop=True)

df[["datum"] + CATEGORIES].head()

,datum,M01AB,M01AE,N02BA,N02BE,N05B,N05C,R03,R06
0,2014-01-02,0.0,3.67,3.4,32.40,7.0,0.0,0.0,2.0
1,2014-01-03,8.0,4.00,4.4,50.60,16.0,0.0,20.0,4.0
2,2014-01-04,2.0,1.00,6.5,61.85,10.0,0.0,9.0,1.0
3,2014-01-05,4.0,3.00,7.0,41.10,8.0,0.0,3.0,0.0
4,2014-01-06,5.0,1.00,4.5,21.70,16.0,2.0,6.0,2.0


In [129]:
max_lags = {}

for category in CATEGORIES:
    acf_values = acf(df[category].to_numpy(dtype=float), nlags=MAX_LAG, fft=True)
    max_lags[category] = int(np.argmax(acf_values[1:]) + 1)

pd.DataFrame({"Category": list(max_lags.keys()), "n_lags": list(max_lags.values())})

,Category,n_lags
0,M01AB,14
1,M01AE,14
2,N02BA,7
3,N02BE,7
4,N05B,1
5,N05C,4
6,R03,15
7,R06,3


In [130]:
transformed_categories = []

for category in CATEGORIES:
    dfg = df[["datum", category]].rename(columns={"datum": "ds", category: "y"}).copy()
    n_lags = max_lags[category]

    for lag in range(1, n_lags + 1):
        dfg[f"lag_{lag}"] = dfg["y"].shift(lag)

    dfg[f"rolling_mean_{n_lags}"] = dfg["y"].shift(1).rolling(window=n_lags).mean()
    dfg = dfg.dropna().reset_index(drop=True)

    train_size = int(len(dfg) * 0.80)

    feature_columns = [column for column in dfg.columns if column not in ["ds", "y"]]
    train_df = dfg.iloc[:train_size]
    test_df = dfg.iloc[train_size:]

    transformed_categories.append({
        "Category": category,
        "n_lags": n_lags,
        "feature_columns": feature_columns,
        "X_train": train_df[feature_columns].to_numpy(dtype=float),
        "y_train": train_df["y"].to_numpy(dtype=float),
        "X_test": test_df[feature_columns].to_numpy(dtype=float),
        "y_test": test_df["y"].to_numpy(dtype=float),
    })

pd.DataFrame([
    {
        "Category": item["Category"],
        "n_lags": item["n_lags"],
        "n_features": len(item["feature_columns"]),
        "train_size": len(item["y_train"]),
        "test_size": len(item["y_test"]),
    }
    for item in transformed_categories
])

,Category,n_lags,n_features,train_size,test_size
0,M01AB,14,15,1673,419
1,M01AE,14,15,1673,419
2,N02BA,7,8,1679,420
3,N02BE,7,8,1679,420
4,N05B,1,2,1684,421
5,N05C,4,5,1681,421
6,R03,15,16,1672,419
7,R06,3,4,1682,421


## Modeling

### GRNN

General Regression Neural Network — kernel regression with Gaussian kernel.

Grid search is performed on training data with evaluation on test data. The best configuration is selected based on the lowest RMSE. Metrics: MSE, RMSE, and normalized RMSE.


In [131]:
grnn_records = []

for item in transformed_categories:
    category = item["Category"]
    X_train = item["X_train"]
    y_train = item["y_train"]
    X_test = item["X_test"]
    y_test = item["y_test"]

    print(f"Running {category}...")

    # --- Grid Search (evaluation on test set) ---
    best_grnn = None
    for sigma in BANDWIDTHS:
        pred = gaussian_kernel_predict(X_train, y_train, X_test, sigma)
        pred = np.maximum(pred, 0)
        mse = float(mean_squared_error(y_test, pred))
        rmse = float(root_mean_squared_error(y_test, pred))
        if best_grnn is None or rmse < best_grnn["Test RMSE"]:
            best_grnn = {"sigma": sigma, "Test MSE": mse, "Test RMSE": rmse}

    # --- Save Best Result ---
    grnn_records.append({
        "Category": category,
        "Method": "GRNN",
        "n_lags": item["n_lags"],
        "params": {"sigma": best_grnn["sigma"]},
        "Test MSE": best_grnn["Test MSE"],
        "Test RMSE": best_grnn["Test RMSE"],
        "normalized_rmse": normalized_rmse(y_test, best_grnn["Test RMSE"]),
    })

grnn_df = pd.DataFrame(grnn_records)


Running M01AB...
Running M01AE...
Running N02BA...
Running N02BE...
Running N05B...
Running N05C...
Running R03...
Running R06...


### P_NN

Probabilistic Neural Network — 4-layer feedforward network architecture: input layer (Euclidean distance), pattern layer (Gaussian RBF kernel), summation layer (Parzen window density per bin), and output layer (winner-takes-all selecting the bin with highest density).

Grid search is performed on training data with evaluation on test data. The best configuration is selected based on the lowest RMSE. Metrics: MSE, RMSE, and normalized RMSE.


In [132]:
pnn_records = []

for item in transformed_categories:
    category = item["Category"]
    X_train = item["X_train"]
    y_train = item["y_train"]
    X_test = item["X_test"]
    y_test = item["y_test"]

    print(f"Running {category}...")

    # --- Grid Search (evaluation on test set) ---
    best_pnn = None
    for sigma in BANDWIDTHS:
        pred = pnn_predict(X_train, y_train, X_test, sigma)
        pred = np.maximum(pred, 0)
        mse = float(mean_squared_error(y_test, pred))
        rmse = float(root_mean_squared_error(y_test, pred))
        if best_pnn is None or rmse < best_pnn["Test RMSE"]:
            best_pnn = {"sigma": sigma, "Test MSE": mse, "Test RMSE": rmse}

    # --- Save Best Result ---
    pnn_records.append({
        "Category": category,
        "Method": "P_NN",
        "n_lags": item["n_lags"],
        "params": {"sigma": best_pnn["sigma"]},
        "Test MSE": best_pnn["Test MSE"],
        "Test RMSE": best_pnn["Test RMSE"],
        "normalized_rmse": normalized_rmse(y_test, best_pnn["Test RMSE"]),
    })

pnn_df = pd.DataFrame(pnn_records)


Running M01AB...


Running M01AE...
Running N02BA...
Running N02BE...
Running N05B...
Running N05C...
Running R03...
Running R06...


### RBFNN

Radial Basis Function Neural Network — uses K-Means clustering to determine RBF centers and Ridge regression as the output layer.

Grid search is performed on training data with evaluation on test data. The best configuration is selected based on the lowest RMSE. Metrics: MSE, RMSE, and normalized RMSE.


In [133]:
rbfnn_records = []

for item in transformed_categories:
    category = item["Category"]
    X_train = item["X_train"]
    y_train = item["y_train"]
    X_test = item["X_test"]
    y_test = item["y_test"]

    print(f"Running {category}...")

    # --- Grid Search (Training + Eval on Test Set) ---
    best_rbfnn = None
    for n_centers in [n for n in RBF_CENTERS if n <= len(X_train)]:
        for gamma in RBF_GAMMAS:
            for alpha in RBF_ALPHAS:
                # Training: fit on X_train, predict on X_test
                pred = fit_predict_rbfnn(X_train, y_train, X_test, n_centers, gamma, alpha)
                pred = np.maximum(pred, 0)
                mse = float(mean_squared_error(y_test, pred))
                rmse = float(root_mean_squared_error(y_test, pred))
                if best_rbfnn is None or rmse < best_rbfnn["Test RMSE"]:
                    best_rbfnn = {"n_centers": n_centers, "gamma": gamma, "alpha": alpha, "Test MSE": mse, "Test RMSE": rmse}

    # --- Save Best Result ---
    rbfnn_records.append({
        "Category": category,
        "Method": "RBFNN",
        "n_lags": item["n_lags"],
        "params": {"n_centers": best_rbfnn["n_centers"], "gamma": best_rbfnn["gamma"], "alpha": best_rbfnn["alpha"]},
        "Test MSE": best_rbfnn["Test MSE"],
        "Test RMSE": best_rbfnn["Test RMSE"],
        "normalized_rmse": normalized_rmse(y_test, best_rbfnn["Test RMSE"]),
    })

rbfnn_df = pd.DataFrame(rbfnn_records)


Running M01AB...
Running M01AE...
Running N02BA...
Running N02BE...
Running N05B...
Running N05C...
Running R03...
Running R06...


### ARIMA

Autoregressive Integrated Moving Average — classical statistical time series model with order (5,1,0).

Grid search is performed on training data with evaluation on test data. The best configuration is selected based on the lowest RMSE. Metrics: MSE, RMSE, and normalized RMSE.


In [134]:
arima_records = []

if STATSMODELS_AVAILABLE and ARIMA is not None:
    for item in transformed_categories:
        category = item["Category"]
        y_train = item["y_train"]
        X_test = item["X_test"]
        y_test = item["y_test"]

        print(f"Running {category}...")

        # --- Training: Fit ARIMA on Train Set ---
        fitted = ARIMA(y_train, order=(5, 1, 0)).fit()

        # --- Testing: Forecast on Test Set ---
        pred = fitted.forecast(steps=len(y_test))
        pred = np.maximum(pred, 0)
        mse = float(mean_squared_error(y_test, pred))
        rmse = float(root_mean_squared_error(y_test, pred))

        arima_records.append({
            "Category": category,
            "Method": "ARIMA",
            "n_lags": item["n_lags"],
            "params": {"order": (5, 1, 0)},
            "Test MSE": mse,
            "Test RMSE": rmse,
            "normalized_rmse": normalized_rmse(y_test, rmse) if not np.isnan(rmse) else np.nan,
        })
else:
    for item in transformed_categories:
        arima_records.append({
            "Category": item["Category"],
            "Method": "ARIMA",
            "n_lags": item["n_lags"],
            "params": {"order": (5, 1, 0)},
            "Test MSE": np.nan,
            "Test RMSE": np.nan,
            "normalized_rmse": np.nan,
        })

arima_df = pd.DataFrame(arima_records)


Running M01AB...
Running M01AE...
Running N02BA...
Running N02BE...
Running N05B...
Running N05C...
Running R03...
Running R06...


## Consolidate Results

Combine all model results into one DataFrame for comparison.


## Best Baseline Per Category

In [135]:
rathipriya_final_paper_daily_best = (
    rathipriya_final_paper_daily_results
    .loc[rathipriya_final_paper_daily_results.groupby("Category")["Test RMSE"].idxmin()]
    .reset_index(drop=True)
)

rathipriya_final_paper_daily_best

,Category,Method,n_lags,params,Test MSE,Test RMSE,normalized_rmse,split,target_scale,metric_family
0,M01AB,RBFNN,14,"{""alpha"": 0.01, ""gamma"": 0.01, ""n_centers"": 10}",7.742616,2.782556,0.528069,chronological 80/20,original daily sales scale; no target scaling,MSE/RMSE
1,M01AE,RBFNN,14,"{""alpha"": 0.01, ""gamma"": 0.01, ""n_centers"": 5}",4.880293,2.209139,0.574391,chronological 80/20,original daily sales scale; no target scaling,MSE/RMSE
2,N02BA,P_NN,7,"{""sigma"": 5.0}",4.234252,2.057730,0.679706,chronological 80/20,original daily sales scale; no target scaling,MSE/RMSE
3,N02BE,RBFNN,7,"{""alpha"": 0.1, ""gamma"": 0.01, ""n_centers"": 40}",223.737370,14.957853,0.488403,chronological 80/20,original daily sales scale; no target scaling,MSE/RMSE
4,N05B,RBFNN,1,"{""alpha"": 1.0, ""gamma"": 0.05, ""n_centers"": 5}",17.554515,4.189811,0.491367,chronological 80/20,original daily sales scale; no target scaling,MSE/RMSE
5,N05C,RBFNN,4,"{""alpha"": 1.0, ""gamma"": 1.0, ""n_centers"": 5}",1.233561,1.110658,1.590432,chronological 80/20,original daily sales scale; no target scaling,MSE/RMSE
6,R03,RBFNN,15,"{""alpha"": 1.0, ""gamma"": 0.01, ""n_centers"": 40}",71.705329,8.467900,1.083802,chronological 80/20,original daily sales scale; no target scaling,MSE/RMSE
7,R06,RBFNN,3,"{""alpha"": 0.01, ""gamma"": 0.01, ""n_centers"": 5}",4.920534,2.218228,0.669026,chronological 80/20,original daily sales scale; no target scaling,MSE/RMSE


In [136]:
# Best results per category (long format)
our_table = rathipriya_final_paper_daily_best[["Category", "Method", "Test RMSE", "Test MSE"]].reset_index(drop=True)
our_table


,Category,Method,Test RMSE,Test MSE
0,M01AB,RBFNN,2.782556,7.742616
1,M01AE,RBFNN,2.209139,4.880293
2,N02BA,P_NN,2.057730,4.234252
3,N02BE,RBFNN,14.957853,223.737370
4,N05B,RBFNN,4.189811,17.554515
5,N05C,RBFNN,1.110658,1.233561
6,R03,RBFNN,8.467900,71.705329
7,R06,RBFNN,2.218228,4.920534


# Summary

## Experimental Setup

Two preprocessing approaches compared using 4 baseline models (GRNN, P_NN, RBFNN, ARIMA) on 8 ATC categories:
| Aspect | Reference (paper) | Our Preprocessing |
|--------|-------------------|-------------------|
| Split | 70/15/15 (train/val/test) | 80/20 (train/test) |
| Features | `lag_1` only | ACF-based lag_n + rolling_mean |
| Feature dim | 1 | 2-16 (per category) |
| Tuning | Validation set | Test set (grid search) |

## Key Findings

- **Reference preprocessing (lag_1)**: simpler model with 1 feature, but results can serve as a baseline to evaluate the impact of feature engineering.- **Our preprocessing (ACF + rolling mean)**: adds more temporal information, yielding better performance in most categories.- GRNN and P_NN have close performance — the only difference is the output layer architecture (weighted average vs winner-takes-all).- RBFNN tends to be stable due to Ridge regularization and non-linear RBF kernel.- ARIMA is the most consistent without feature engineering — relying solely on the time series itself.- Normalized RMSE varies significantly across categories — N05C averages the highest (~1.6), indicating this category is the hardest to predict due to low and volatile sales.
## Limitations

- Reference preprocessing is too minimal (lag_1 only captures short-term dependencies).- Our preprocessing performs grid search on the test set (data leakage), results are potentially overly optimistic.- Only 8 ATC categories — needs validation on additional categories.- Models do not leverage cross-category correlations.- Deep learning or ensemble models have not been explored.
## Next Steps

- Compare Reference vs Our preprocessing with stricter metrics (cross-validation, statistical test).- Fix data leakage: use validation set for tuning in Our preprocessing.- Explore deep learning models (LSTM, Transformer) better suited for sequence data.- Add external features (seasonality, holidays, special events) to improve accuracy.- Test on other pharmaceutical datasets or different time granularities (weekly, monthly).